In [7]:
import os, sys, shutil
from pathlib import Path
try:
    from google.colab import userdata, drive
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
except Exception:
    import getpass
    os.environ['OPENAI_API_KEY'] = getpass.getpass('OPENAI_API_KEY: ')
if not os.path.exists('/content/mmai'):
    !git clone https://github.com/michaelyserrano/mmai.git /content/mmai
    !pip install -q -r /content/mmai/project/requirements.txt
%cd /content/mmai
sys.path.insert(0, 'project')
drive.mount('/content/drive')
DRIVE_TB = Path('/content/drive/MyDrive/Colab Notebooks/toolbench_data')
LOCAL_TB = Path('/content/toolbench_data')
if not LOCAL_TB.exists():
    print('Copying toolbench to local disk (first time, slow)...')
    !cp -r "{DRIVE_TB}" "{LOCAL_TB}"
else:
    for fname in ['toolllama_G123_dfs_eval.json', 'toolllama_G123_dfs_train.json']:
        if not (LOCAL_TB / fname).exists():
            shutil.copy(DRIVE_TB / fname, LOCAL_TB / fname)
            print(f'Copied missing {fname}')
TOOLBENCH_DIR = LOCAL_TB

/content/mmai
Using existing local toolbench data.


In [8]:
!git -C /content/mmai pull
from data.load_toolbench import load_api_corpus
from models.embeddings import format_api_string, get_embeddings
import numpy as np
from tqdm import tqdm
from pathlib import Path

corpus = load_api_corpus(Path(TOOLBENCH_DIR) / "toolenv" / "tools")
print(f"Loaded {len(corpus)} APIs across {len(set(a['category'] for a in corpus))} categories")

Already up to date.
Loaded 2733 APIs across 4 categories


In [9]:
api_strings = [format_api_string(a) for a in corpus]
print("Sample:", api_strings[0])

# Embed all APIs (cached — safe to re-run)
BATCH = 500
all_embeddings = []
for i in tqdm(range(0, len(api_strings), BATCH)):
    batch = api_strings[i:i+BATCH]
    embs = get_embeddings(batch)
    all_embeddings.append(embs)

corpus_matrix = np.vstack(all_embeddings)
np.save("project/corpus_embeddings.npy", corpus_matrix)
print(f"✓ Saved corpus_embeddings.npy: shape {corpus_matrix.shape}")

Sample: Devices > Test: test — 1 | Parameters: s


100%|██████████| 6/6 [00:00<00:00, 14.49it/s]

✓ Saved corpus_embeddings.npy: shape (2733, 1536)


In [10]:
import json
api_names = [a["action_name"] for a in corpus]
with open("project/api_names.json", "w") as f:
    json.dump(api_names, f)
print(f"✓ Saved {len(api_names)} API names")

✓ Saved 2733 API names


In [11]:
import json, shutil
from pathlib import Path
api_names = [a['name'] for a in corpus]
with open('project/api_names.json', 'w') as f:
    json.dump(api_names, f)
drive_out = Path('/content/drive/MyDrive/Colab Notebooks/mmai/project')
drive_out.mkdir(parents=True, exist_ok=True)
shutil.copy('project/corpus_embeddings.npy', drive_out / 'corpus_embeddings.npy')
shutil.copy('project/api_names.json', drive_out / 'api_names.json')
print(f'Saved to Drive: {drive_out}')

Saved to Drive: /content/drive/MyDrive/Colab Notebooks/mmai/project
